# False Prediction Analysis

Extracts all samples where the model predicted wrong for each Big Five trait,
and writes 6 markdown files into the result directory alongside `predictions.csv`:

| File | Contents |
|---|---|
| `false_Openness.md` | All samples wrong on Openness |
| `false_Conscientiousness.md` | All samples wrong on Conscientiousness |
| `false_Extraversion.md` | All samples wrong on Extraversion |
| `false_Agreeableness.md` | All samples wrong on Agreeableness |
| `false_Neuroticism.md` | All samples wrong on Neuroticism |
| `false_all5.md` | Samples wrong on **all 5 traits** |

Each entry shows the input text and the full LLM reasoning response.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "extract_false_predictions.py").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from extract_false_predictions import extract_false_predictions

print("Project root:", project_root)

## Configuration

In [ ]:
# ── point these at the run you want to analyse ──────────────────────────────
model_name  = "gpt-4o-mini"
prompt_mode = "reasoned_rag_def_oneshot_30f"
run_id      = "20260523-221029"

# ── derived paths (no need to edit below this line) ─────────────────────────
log_path  = project_root / "log"  / model_name / prompt_mode / f"{run_id}_log.txt"
pred_path = project_root / "result" / model_name / prompt_mode / run_id / "predictions.csv"

# ── output directory (default: same folder as predictions.csv) ───────────────
out_dir = None   # set to a Path or string to override, e.g. project_root / "analysis"

# ── max samples written per file (247 = write all) ───────────────────────────
max_per_file = 247

print(f"Log  : {log_path}")
print(f"Pred : {pred_path}")
print(f"  log  exists: {log_path.exists()}")
print(f"  pred exists: {pred_path.exists()}")

## Run

In [ ]:
results = extract_false_predictions(
    log_path     = log_path,
    pred_path    = pred_path,
    out_dir      = out_dir,
    max_per_file = max_per_file,
)

## Results summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {"trait": trait, "wrong_predictions": n}
    for trait, n in results["per_trait"].items()
])
summary.loc[len(summary)] = {"trait": "All 5 wrong (samples)", "wrong_predictions": results["all5_count"]}
display(summary)

print(f"\nOutput files written to: {results['out_dir']}")